In [10]:
from pystlouisfed import FRED
import os
import numpy as np
import pandas as pd
import yfinance as yf

In [3]:
key = os.getenv("FRED_API_KEY")
fred = FRED(api_key=key)

In [27]:
cpi = fred.series_observations("CPIAUCSL", observation_start="1968-01-01")["value"].astype(float)
cpi.index = cpi.index.to_period("M")
infl_m = cpi.pct_change()
infl_yoy = cpi.pct_change(12)


tbill = fred.series_observations("TB3MS", observation_start="1968-01-01")["value"].astype(float) / 100 / 12
tbill.index = tbill.index.to_period("M")

In [28]:
tbill

date
1968-01    0.004167
1968-02    0.004150
1968-03    0.004308
1968-04    0.004483
1968-05    0.004717
             ...   
2026-05    0.003000
2026-06    0.003050
2026-07    0.003108
2026-08    0.003100
2026-09    0.003283
Freq: M, Name: value, Length: 705, dtype: float64

In [29]:
cpi

date
1968-01     34.100
1968-02     34.200
1968-03     34.300
1968-04     34.400
1968-05     34.500
            ...   
2026-05    333.979
2026-06    332.568
2026-07    332.813
2026-08    334.131
2026-09    336.298
Freq: M, Name: value, Length: 705, dtype: float64

In [36]:
sp = yf.download("^SP500TR", start="1987-12-01", interval="1mo",
                 auto_adjust=False, progress=False)["Close"].squeeze()
sp.index = sp.index.to_period("M")
eq_ret = sp.pct_change()

In [37]:
gsci = yf.download("^SPGSCI", start="1969-01-01", interval="1mo",
                   auto_adjust=False, progress=False)["Close"].squeeze()
gsci.index = gsci.index.to_period("M")


In [38]:
gsci

Date
1985-01    193.779999
1985-02    188.229996
1985-03    197.960007
1985-04    187.419998
1985-05    187.619995
              ...    
2026-06    619.450012
2026-07    686.150024
2026-08    716.700012
2026-09    730.419983
2026-10    754.409973
Freq: M, Name: ^SPGSCI, Length: 502, dtype: float64

In [39]:
com_ret = gsci.pct_change() + tbill

df = pd.concat({
    "cpi": cpi,
    "infl_m": infl_m,
    "infl_yoy": infl_yoy,
    "tbill": tbill,
    "equity": eq_ret,
    "commodity": com_ret,
}, axis=1).loc["1970-01":].dropna()

df

,cpi,infl_m,infl_yoy,tbill,equity,commodity
1988-02,116.200,0.001724,0.039356,0.004717,0.046646,-0.012327
1988-03,116.500,0.002582,0.038324,0.004750,-0.030911,0.049991
1988-04,117.200,0.006009,0.039929,0.004925,0.011066,0.027180
1988-05,117.500,0.002560,0.039823,0.005217,0.008635,0.022244
1988-06,118.000,0.004255,0.039648,0.005383,0.045885,-0.032372
...,...,...,...,...,...,...
2026-05,333.979,0.004729,0.041666,0.003000,0.052633,-0.093118
2026-06,332.568,-0.004225,0.034635,0.003050,-0.009523,-0.105103
2026-07,332.813,0.000737,0.033039,0.003108,-0.000635,0.110785
2026-08,334.131,0.003960,0.033530,0.003100,0.027233,0.047624
